TIENE 4 CIFRE SIGNIFICATIVE NEL PRIMO E 5 IN TUTTE LE ALTRE + RIORDINO TRAJ_ID

In [1]:
import pandas as pd
import os

def process_csv_advanced(input_file, output_file, duration_threshold):
    # 1. Leggi il file CSV
    # Se il CSV originale ha virgole come separatori, usa sep=','
    df = pd.read_csv(input_file)
    
    # Identifica le colonne (assumendo index 0 = timestamp, index 1 = traj_id)
    ts_col = df.columns[0]
    traj_col = df.columns[1]
    
    # 2. Filtra le traj_id basandoti sul conteggio delle occorrenze
    counts = df[traj_col].value_counts()
    valid_ids = counts[counts >= duration_threshold].index
    df = df[df[traj_col].isin(valid_ids)].copy()
    
    # 3. Traslazione dei traj_id (Rinumero continuo)
    # Ordiniamo gli ID rimanenti per garantire la sequenza corretta
    unique_ids = sorted(df[traj_col].unique())
    
    # Creiamo un mapping: 
    # Manteniamo il primo ID come punto di partenza, poi incrementiamo di 1
    # Esempio: [22, 24, 25] -> {22: 22, 24: 23, 25: 24}
    start_id = unique_ids[0]
    mapping = {old: start_id + i for i, old in enumerate(unique_ids)}
    
    # Applichiamo il mapping alla colonna
    df[traj_col] = df[traj_col].map(mapping)
    
    # 4. Processamento Timestamp
    # Sottraiamo il primo timestamp (che ora è il valore minimo della colonna timestamp)
    first_timestamp = df.iloc[0, 0]
    
    # Formattazione colonna 0 (timestamp) -> 4 decimali e virgola
    df[ts_col] = (df[ts_col] - first_timestamp).apply(lambda x: f"{x:.4f}".replace('.', ','))
    
    # 5. Formattazione colonne restanti -> 5 decimali e virgola
    for col in df.columns[2:]:
        df[col] = df[col].apply(lambda x: f"{x:.5f}".replace('.', ','))
    
    # 6. Salvataggio
    # Usiamo sep=';' come richiesto e indichiamo decimal=',' per coerenza
    df.to_csv(output_file, index=False, sep=';')
    print(f"File elaborato correttamente e salvato come: {output_file}")

# Esempio di utilizzo:
# Imposta la soglia desiderata (es. 10 righe minime)

input_file = "C:\\Users\\Carlo\\time-series-autoencoder\\examples\\reconstruction\\NUOVI DATI RACCOLTI\\QUERY_CSV_SANE2.csv"

# 1. Separa il percorso (cartella) dal nome del file
directory, filename = os.path.split(input_file)
# 2. Separa il nome del file dall'estensione (.csv)
name_without_ext, ext = os.path.splitext(filename)
# 3. Crea il nuovo nome aggiungendo il suffisso
new_filename = f"{name_without_ext}_EXCEL{ext}"

# 4. Ricostruisci il percorso completo
output_file = os.path.join(directory, new_filename)

print(f"File di output: {output_file}")

threshold = 127 

process_csv_advanced(input_file, output_file, threshold)

File di output: C:\Users\Carlo\time-series-autoencoder\examples\reconstruction\NUOVI DATI RACCOLTI\QUERY_CSV_SANE2_EXCEL.csv
File elaborato correttamente e salvato come: C:\Users\Carlo\time-series-autoencoder\examples\reconstruction\NUOVI DATI RACCOLTI\QUERY_CSV_SANE2_EXCEL.csv
